# Memoria negli Agenti AI con `mem0`

Gli agenti AI senza memoria soffrono di **amnesia**: ogni conversazione ricomincia da zero, l'agente non ricorda chi sei, cosa hai fatto, cosa hai chiesto prima.

In questo notebook aggiungiamo **memoria persistente** a un agente usando [`mem0`](https://mem0.ai): un layer che estrae, indicizza e recupera ricordi in modo semantico.

```
┌─────────────────────────────────────────────────────┐
│              AGENTE CON MEMORIA                     │
│                                                     │
│   Input → [CERCA ricordi] → THINK → ACT → OBSERVE  │
│               ↑                            │        │
│               └──── [SALVA ricordi] ←──────┘        │
└─────────────────────────────────────────────────────┘
```

**Stack usato:**
| Componente | Libreria | Note |
|-----------|---------|------|
| LLM | `huggingface_hub` `InferenceClient` | Serverless, API HF compatibile OpenAI |
| Embedder | `sentence-transformers` `paraphrase-multilingual-MiniLM-L12-v2` | Multilingua, gira su CPU |
| Vector store | `qdrant-client` in-memory | Nessun server da avviare |
| Memoria | `mem0` | Estrae e recupera ricordi semantici |

## 1. Installazione delle Dipendenze

Se stai usando l'ambiente `uv` di questo progetto, ricorda di aggiungere le dipendenze:
```bash
uv add mem0ai sentence-transformers qdrant-client openai
```

Se sei su **Colab** o un ambiente diverso, decommentla cella qui sotto:

In [ ]:
# Decommentare solo se NON si usa l'ambiente uv del progetto
# %pip install mem0ai sentence-transformers qdrant-client openai huggingface_hub python-dotenv

## 2. Import delle Librerie

In [1]:
import os
import json
import time

# Evita lock locali di Qdrant quando si rieseguono le celle nel notebook
os.environ["MEM0_TELEMETRY"] = "False"

from dotenv import load_dotenv
from huggingface_hub import InferenceClient
from mem0 import Memory

print("Librerie importate correttamente!")

Librerie importate correttamente!


## 3. Autenticazione HuggingFace

Serve un HF token (tier gratuito è sufficiente):
1. Vai su https://hf.co/settings/tokens
2. Crea un token con permessi `write`
3. Inseriscilo nel file `.env`: `HF_TOKEN=hf_tuoTokenQui`

In [2]:
load_dotenv(".env")

HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    print(f"Token HF caricato: {HF_TOKEN[:8]}...")
else:
    print("ATTENZIONE: HF_TOKEN non trovato nel file .env")

Token HF caricato: hf_GxbIC...


## 4. Configurazione

Definiamo i modelli da usare:

- **`Qwen2.5-7B-Instruct`**: uno dei più affidabili per tool calling sul tier gratuito HF
- **`paraphrase-multilingual-MiniLM-L12-v2`**: embedder leggero multilingua (384 dimensioni), adatto a query in italiano

Ogni utente ha un `user_id` che isola i suoi ricordi dagli altri.

In [3]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" # "meta-llama/Llama-3.2-3B-Instruct"

# Embedder locale multilingua — 384 dimensioni, CPU-friendly, nessuna chiamata API
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Soglia bassa: la demo usa query italiane e mem0 può estrarre ricordi in inglese
MEMORY_SEARCH_THRESHOLD = 0.0

# ID dell'utente — mem0 usa questo per isolare i ricordi per persona
USER_ID = "mario.rossi"

print(f"LLM: {MODEL_ID}")
print(f"Embedder: {EMBED_MODEL}")
print(f"User ID: {USER_ID}")

LLM: Qwen/Qwen2.5-7B-Instruct
Embedder: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
User ID: mario.rossi


## 5. Inizializzare Mem0

**Come funziona mem0:**
1. `memory.add(messages, user_id=...)` → chiama l'LLM per estrarre i fatti rilevanti dal dialogo e li indicizza come vettori
2. `memory.search(query, filters={"user_id": ...})` → converte la query in vettore e recupera i ricordi più simili

**Nota API mem0 2.x:** `add()` e `delete_all()` usano `user_id=...`; `search()` e `get_all()` usano invece `filters={"user_id": ...}`.

**Trucco chiave:** mem0 accetta qualsiasi endpoint OpenAI-compatibile configurandolo come provider `"openai"` con `openai_base_url` custom. HuggingFace espone l'endpoint OpenAI-compatible su `https://router.huggingface.co/v1`.

**Qdrant in-memory:** passando `"path": ":memory:"` nel config, il vector store vive solo in RAM — nessun server da avviare, si svuota al riavvio del kernel.

In [4]:
mem0_config = {
    # LLM: usa HF Inference Providers come endpoint OpenAI-compatible
    "llm": {
        "provider": "openai",
        "config": {
            "model": MODEL_ID,
            "openai_base_url": "https://router.huggingface.co/v1",
            "api_key": HF_TOKEN,
            "temperature": 0.1,
            "max_tokens": 800,
        },
    },
    # Embedder: sentence-transformers locale, nessuna API key necessaria
    "embedder": {
        "provider": "huggingface",
        "config": {
            "model": EMBED_MODEL,
        },
    },
    # Vector store: Qdrant completamente in RAM
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "collection_name": "memory_demo",
            "embedding_model_dims": 384,  # dimensioni del MiniLM multilingua
            "path": ":memory:",           # nessun server, tutto in RAM
        },
    },
    # Anche la history SQLite resta volatile: demo pulita a ogni kernel
    "history_db_path": ":memory:",
}

print("Inizializzazione mem0 in corso (download embedder al primo avvio)...")
memory = Memory.from_config(mem0_config)
print("Mem0 pronto!")

Inizializzazione mem0 in corso (download embedder al primo avvio)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Mem0 pronto!


/Users/giovannibonetta/Desktop/progetti/agent_demo/.venv/lib/python3.12/site-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()


## 6. Definire i Tool dell'Agente

Usiamo il formato **OpenAI tool calling** (JSON Schema), compatibile con `InferenceClient.chat_completion`.

Definiamo due tool:
- **`calculate`**: valuta espressioni matematiche Python
- **`get_user_profile`**: recupera manualmente informazioni sull'utente dalla memoria (tool esplicito, utile per debug)

In [5]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Calcola un'espressione matematica Python e restituisce il risultato numerico.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "L'espressione da valutare, es. '1234 * 5678' o '(100 + 50) / 3'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_user_profile",
            "description": "Recupera le informazioni memorizzate sull'utente corrente dalla memoria a lungo termine.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Cosa cercare nella memoria, es. 'nome utente', 'ruolo lavorativo', 'preferenze'",
                    }
                },
                "required": ["query"],
            },
        },
    },
]


def execute_tool(name: str, args: dict, user_id: str) -> str:
    """Dispatcher: esegue il tool richiesto e restituisce il risultato come stringa."""
    if name == "calculate":
        try:
            allowed = set("0123456789+-*/()., ")
            expr = args["expression"]
            if not all(c in allowed for c in expr):
                return "Errore: solo operazioni matematiche base permesse."
            result = eval(expr)
            return f"{expr} = {result}"
        except Exception as e:
            return f"Errore nel calcolo: {e}"

    elif name == "get_user_profile":
        results = memory.search(
            args["query"],
            filters={"user_id": user_id},
            top_k=3,
            threshold=MEMORY_SEARCH_THRESHOLD,
        )
        if not results or not results.get("results"):
            return "Nessuna informazione trovata in memoria."
        items = results["results"]
        return "\n".join(f"- {m['memory']}" for m in items)

    return f"Tool '{name}' non trovato."


print(f"Tool definiti: {[t['function']['name'] for t in TOOLS]}")

Tool definiti: ['calculate', 'get_user_profile']


## 7. Creare il Client LLM

Usiamo un **client separato** per l'agente (con temperatura più alta per essere più conversazionale) rispetto a quello che mem0 usa internamente (temperatura 0.1 per estrarre fatti in modo preciso).

In [6]:
client = InferenceClient(
    model=MODEL_ID,
    token=HF_TOKEN,
    provider="auto",
    timeout=60,
)

print(f"Client LLM pronto: {MODEL_ID}")

Client LLM pronto: Qwen/Qwen2.5-7B-Instruct


## 8. L'Agent Loop con Memoria

La funzione `agent_with_memory` implementa il ciclo completo:

```
1. Riceve il messaggio utente
2. Cerca ricordi rilevanti in mem0
3. Inserisce quei ricordi nel system prompt
4. Chiama il modello
5. Se il modello vuole usare tool, li esegue
6. Ripassa i risultati dei tool al modello
7. Ottiene la risposta finale
8. Salva il nuovo messaggio utente in memoria
9. Ritorna la risposta

```

**Nota sul rate limiting:** HF free tier ha limiti di richieste al minuto. Il `time.sleep(1)` tra le iterazioni del loop evita errori 429.

In [7]:
def agent_with_memory(user_message: str, user_id: str, verbose: bool = True) -> str:
    """
    Agente con memoria a lungo termine.
    - Recupera ricordi rilevanti prima di rispondere
    - Salva l'interazione in memoria dopo aver risposto
    """

    # ── STEP 1: Recupera ricordi rilevanti ──────────────────────────────
    search_results = memory.search(
        user_message,
        filters={"user_id": user_id},
        top_k=5,
        threshold=MEMORY_SEARCH_THRESHOLD,
    )
    memories = search_results.get("results", []) if search_results else []
    memory_context = "\n".join(f"- {m['memory']}" for m in memories)

    if verbose and memory_context:
        print(f"[MEMORIA] Ricordi recuperati:\n{memory_context}\n")
    elif verbose:
        print("[MEMORIA] Nessun ricordo trovato per questa query.\n")

    # ── STEP 2: Costruisci il contesto ──────────────────────────────────
    system_prompt = (
        "Sei un assistente utile e preciso. Rispondi in italiano. "
        "Quando riassumi informazioni sull'utente, parla dell'utente: non parlare come se fossi lui."
    )
    if memory_context:
        system_prompt += f"\n\nInformazioni che ricordo di questo utente:\n{memory_context}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    # ── STEP 3: Agent loop con tool calling ─────────────────────────────
    answer = "Limite di iterazioni raggiunto."
    MAX_ITER = 5

    for step in range(MAX_ITER):
        if verbose:
            print(f"[STEP {step + 1}] Chiamo il modello...")

        response = client.chat_completion(
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            max_tokens=800,
            temperature=0.3,
        )

        msg = response.choices[0].message

        # Se l'agente non chiama tool → ha finito
        if not msg.tool_calls:
            answer = msg.content or ""
            if verbose:
                print(f"[STEP {step + 1}] Risposta finale generata.")
            break

        # Esegui i tool call e aggiungi i risultati ai messaggi
        tool_calls_serialized = [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                    if isinstance(tc.function.arguments, str)
                    else json.dumps(tc.function.arguments),
                },
            }
            for tc in msg.tool_calls
        ]
        messages.append(
            {"role": "assistant", "content": None, "tool_calls": tool_calls_serialized}
        )

        for tc in msg.tool_calls:
            args = (
                json.loads(tc.function.arguments)
                if isinstance(tc.function.arguments, str)
                else tc.function.arguments
            )
            result = execute_tool(tc.function.name, args, user_id)
            if verbose:
                print(f"[TOOL] {tc.function.name}({args}) → {result}")
            messages.append(
                {"role": "tool", "tool_call_id": tc.id, "content": result}
            )

        time.sleep(1)  # evita rate limiting HF free tier

    # ── STEP 4: Salva i fatti dichiarati dall'utente in memoria ─────────
    # NOTA: add() usa user_id= direttamente; search/get_all usano filters={"user_id": ...}
    memory.add(
        messages=[{"role": "user", "content": user_message}],
        user_id=user_id,
    )
    if verbose:
        print("[MEMORIA] Interazione salvata.\n")

    return answer


print("Funzione agent_with_memory definita!")

Funzione agent_with_memory definita!


## 9. Demo — Sessione 1: Presentazione

Nella prima interazione l'agente non sa nulla di Mario. Alla fine, mem0 estrarrà e salverà i fatti rilevanti dalla conversazione.

In [8]:
print("=" * 60)
print("SESSIONE 1: Presentazione")
print("=" * 60 + "\n")

risposta = agent_with_memory(
    "Ciao! Mi chiamo Mario Rossi, sono il CFO di Infojuice. "
    "Lavoro principalmente con budget in EUR e preferisco risposte concise.",
    USER_ID,
)

print("\n" + "─" * 60)
print("RISPOSTA:", risposta)

Failed to load spaCy lemma model: spaCy is not installed. Install it with: pip install mem0ai[nlp]
Failed to load spaCy full model: spaCy is not installed. Install it with: pip install mem0ai[nlp]


SESSIONE 1: Presentazione



fastembed not installed — BM25 keyword search disabled. Install with: pip install fastembed


[MEMORIA] Nessun ricordo trovato per questa query.

[STEP 1] Chiamo il modello...
[TOOL] get_user_profile({'query': 'nome utente, ruolo lavorativo, preferenze'}) → Nessuna informazione trovata in memoria.
[STEP 2] Chiamo il modello...
[STEP 2] Risposta finale generata.
[MEMORIA] Interazione salvata.


────────────────────────────────────────────────────────────
RISPOSTA: Mario Rossi è il CFO di Infojuice. Lavora principalmente con budget in EUR e preferisce risposte concise.


## 10. Demo — Sessione 2: La Memoria Funziona?

In questa seconda interazione, l'agente dovrebbe **ricordare** il nome e il ruolo di Mario, anche se non li ha ripetuti.

In [9]:
print("=" * 60)
print("SESSIONE 2: Recall identità")
print("=" * 60 + "\n")

risposta = agent_with_memory(
    "Come mi chiamo e qual è il mio ruolo in azienda?",
    USER_ID,
)

print("\n" + "─" * 60)
print("RISPOSTA:", risposta)

SESSIONE 2: Recall identità

[MEMORIA] Ricordi recuperati:
- User's name is Mario Rossi and he is the CFO of Infojuice
- Mario primarily works with EUR budgets and prefers concise responses

[STEP 1] Chiamo il modello...
[TOOL] get_user_profile({'query': 'nome utente, ruolo lavorativo'}) → - User's name is Mario Rossi and he is the CFO of Infojuice
- Mario primarily works with EUR budgets and prefers concise responses
[STEP 2] Chiamo il modello...
[STEP 2] Risposta finale generata.
[MEMORIA] Interazione salvata.


────────────────────────────────────────────────────────────
RISPOSTA: Il tuo nome è Mario Rossi e sei il CFO di Infojuice.


## 11. Demo — Sessione 3: Memoria + Tool Calling

Qui l'agente deve:
1. Ricordare dalla memoria che Mario è CFO di Infojuice e lavora in EUR
2. Usare il tool `calculate` per eseguire il calcolo

In [10]:
print("=" * 60)
print("SESSIONE 3: Memoria + Tool Calling")
print("=" * 60 + "\n")

risposta = agent_with_memory(
    "Ho un budget di 1234 EUR e devo moltiplicarlo per 5678. Quanto fa?",
    USER_ID,
)

print("\n" + "─" * 60)
print("RISPOSTA:", risposta)

SESSIONE 3: Memoria + Tool Calling

[MEMORIA] Ricordi recuperati:
- Mario primarily works with EUR budgets and prefers concise responses
- User's name is Mario Rossi and he is the CFO of Infojuice

[STEP 1] Chiamo il modello...
[TOOL] calculate({'expression': '1234 * 5678'}) → 1234 * 5678 = 7006652
[STEP 2] Chiamo il modello...
[STEP 2] Risposta finale generata.
[MEMORIA] Interazione salvata.


────────────────────────────────────────────────────────────
RISPOSTA: Il risultato della moltiplicazione di 1234 EUR per 5678 è 7006652 EUR.


## 12. Ispezionare la Memoria

Possiamo vedere esattamente cosa mem0 ha estratto e salvato dalle conversazioni precedenti.

In [11]:
print("=" * 60)
print(f"TUTTI I RICORDI per '{USER_ID}'")
print("=" * 60)

all_memories = memory.get_all(filters={"user_id": USER_ID})
results = all_memories.get("results", []) if all_memories else []

if not results:
    print("Nessun ricordo trovato.")
else:
    for i, m in enumerate(results, 1):
        print(f"\n[{i}] {m['memory']}")
        if "created_at" in m:
            print(f"    Creato: {m['created_at']}")

TUTTI I RICORDI per 'mario.rossi'

[1] User is Mario Rossi, the CFO of Infojuice who primarily works with EUR budgets and prefers concise responses
    Creato: 2026-05-11T19:19:16.918272+00:00

[2] User's name is Mario Rossi and he is the CFO of Infojuice
    Creato: 2026-05-11T19:17:44.780757+00:00

[3] User has a budget of 1234 EUR and needs to multiply it by 5678
    Creato: 2026-05-11T19:19:16.918254+00:00

[4] Mario primarily works with EUR budgets and prefers concise responses
    Creato: 2026-05-11T19:17:44.780789+00:00


In [12]:
# Ricerca semantica nella memoria — recupera i ricordi più rilevanti per una query
print("=" * 60)
print("RICERCA SEMANTICA: 'informazioni lavorative'")
print("=" * 60)

search_results = memory.search(
    "informazioni lavorative",
    filters={"user_id": USER_ID},
    top_k=5,
    threshold=MEMORY_SEARCH_THRESHOLD,
)
results = search_results.get("results", []) if search_results else []

for i, m in enumerate(results, 1):
    score = m.get("score", "n/a")
    print(f"[{i}] (score: {score:.3f}) {m['memory']}")

RICERCA SEMANTICA: 'informazioni lavorative'
[1] (score: 0.351) User is Mario Rossi, the CFO of Infojuice who primarily works with EUR budgets and prefers concise responses
[2] (score: 0.253) User's name is Mario Rossi and he is the CFO of Infojuice
[3] (score: 0.190) Mario primarily works with EUR budgets and prefers concise responses
[4] (score: 0.160) User has a budget of 1234 EUR and needs to multiply it by 5678


## 13. Confronto: Con vs Senza Memoria

Vediamo la differenza concreta tra un agente con memoria e uno senza, usando la stessa domanda.

In [14]:
def agent_without_memory(user_message: str) -> str:
    """Stesso agente, ma senza recuperare né salvare ricordi."""
    messages = [
        {"role": "system", "content": "Sei un assistente utile. Rispondi in italiano."},
        {"role": "user", "content": user_message},
    ]
    response = client.chat_completion(
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
        max_tokens=400,
        temperature=0.1,
    )
    return response.choices[0].message.content or ""


DOMANDA = "Come mi chiamo e qual è il mio ruolo?"

print("=" * 60)
print("DOMANDA:", DOMANDA)
print("=" * 60)

print("\n[SENZA MEMORIA]")
risposta_no_mem = agent_without_memory(DOMANDA)
print(risposta_no_mem)

print("\n[CON MEMORIA]")
risposta_con_mem = agent_with_memory(DOMANDA, USER_ID, verbose=False)
print(risposta_con_mem)

DOMANDA: Come mi chiamo e qual è il mio ruolo?

[SENZA MEMORIA]


[CON MEMORIA]
Il suo nome è Mario Rossi e il suo ruolo è CFO di Infojuice.


## 14. Aggiornamento e Cancellazione dei Ricordi

mem0 gestisce automaticamente gli **aggiornamenti**: se dici "ora lavoro in un'altra azienda", mem0 aggiorna il ricordo precedente invece di duplicarlo.

Possiamo anche cancellare esplicitamente singoli ricordi o tutta la memoria di un utente.

In [15]:
# Aggiornamento automatico: mem0 aggiorna il ricordo esistente
print("=" * 60)
print("TEST: Aggiornamento automatico")
print("=" * 60 + "\n")

risposta = agent_with_memory(
    "Da oggi sono il CEO, non più CFO. Ho cambiato ruolo.",
    USER_ID,
    verbose=False,
)
print("Aggiornamento comunicato:", risposta)

# Verifica: il ricordo dovrebbe riflettere il nuovo ruolo
print("\n── Ricordi dopo l'aggiornamento ──")
results = memory.get_all(filters={"user_id": USER_ID}).get("results", [])
for m in results:
    print(f"  • {m['memory']}")

TEST: Aggiornamento automatico

Aggiornamento comunicato: Ho notato che il tuo ruolo è cambiato da CFO a CEO. Come posso aiutarti nel tuo nuovo ruolo?

── Ricordi dopo l'aggiornamento ──
  • User's role changed from CFO to CEO as of today
  • User is Mario Rossi, the CFO of Infojuice who primarily works with EUR budgets and prefers concise responses
  • User's name is Mario Rossi and he is the CFO of Infojuice
  • User has a budget of 1234 EUR and needs to multiply it by 5678
  • User asked about their name and role
  • Mario primarily works with EUR budgets and prefers concise responses


In [ ]:
# OPZIONALE: cancella tutta la memoria dell'utente
# Decommentare per eseguire

# memory.delete_all(user_id=USER_ID)
# print(f"Tutti i ricordi di '{USER_ID}' cancellati.")

## Riepilogo

| Concetto | Come funziona |
|---------|---------------|
| **mem0** | Estrae fatti dal dialogo con un LLM, li indicizza come vettori |
| **`memory.search()`** | Recupera ricordi semanticamente simili alla query corrente |
| **`memory.add()`** | Salva una nuova interazione, aggiornando ricordi esistenti se in conflitto |
| **HF Inference API** | Endpoint OpenAI-compatibile → configurabile in mem0 come provider `"openai"` |
| **Qdrant in-memory** | Vector store senza server, tutto in RAM |
| **`user_id`** | Isola i ricordi per utente — ogni persona ha il suo spazio di memoria |

**Pattern chiave:**
```
1. memory.search(query)  → context per il system prompt
2. LLM + tools           → risposta
3. memory.add(dialogo)   → aggiorna la memoria
```

Questo schema è indipendente dal framework: funziona ugualmente con `smolagents`, `langchain`, `llama-index`, o un agente custom come quello in questo notebook.